# executorlib

## Python Standard Library

`executorlib` extends the `Executor` interface defined in the [Python Standard Library's `concurrent.futures` module](https://docs.python.org/3/library/concurrent.futures.html). Before diving into `executorlib` itself it is worth understanding the standard-library executors it is built on.

The standard library defines two ready-made executor classes:
* `ThreadPoolExecutor` — runs Python functions in separate **threads** within the same process. Fast to start, but limited by Python's Global Interpreter Lock (GIL) for CPU-bound work.
* `ProcessPoolExecutor` — runs Python functions in separate **processes**. Circumvents the GIL for true CPU parallelism, but adds the overhead of spawning processes and serialising data.

**What executorlib adds on top:**
Both standard-library executors are limited to a single machine. `executorlib` replaces the process-spawning backend with calls to an HPC job scheduler (SLURM or Flux), so the same `submit()` / `Future` API you already know scales from a laptop to thousands of HPC cores *without changing your Python code*.

**Learning objectives:** After this notebook you will be able to:
- Use `Future` objects to write non-blocking, asynchronous Python workflows
- Select the right `executorlib` executor class for your use case
- Assign per-function computing resources (cores, GPUs, memory) with the `resource_dict`
- Use caching to avoid redundant computation
- Scale a serial workflow to HPC by swapping one class name

---

Let's start with the simple loop that we want to parallelise:


In [1]:
result_lst = []
for i in range(1, 5):
    result_lst.append(sum([i, i]))

result_lst

[2, 4, 6, 8]

This can be rewritten using either the `ThreadPoolExecutor` or the `ProcessPoolExecutor`. In both cases we use a `with` statement to start the executor and shut it down once all tasks are finished. The `with` block ensures that background workers are cleaned up correctly even if an exception occurs — just as `with open(...)` ensures a file handle is closed regardless of errors.

The key change from the plain loop is that `exe.submit(fn, *args)` **returns immediately** with a [`concurrent.futures.Future`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Future) object rather than the actual result. The function runs in the background while the main thread continues. Calling `.result()` on the `Future` blocks until the result is available.


In [2]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[True, True, True, False]
[2, 4, 6, 8]


In [3]:
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


### `Future` objects and asynchronous execution

A [`concurrent.futures.Future`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Future) represents the **eventual result** of an asynchronous computation. The most important methods are:

| Method | Description |
|---|---|
| `f.result()` | Block and return the result (or raise the exception if the function failed) |
| `f.done()` | Return `True` if the function has finished (without blocking) |
| `f.cancel()` | Attempt to cancel a pending function (not always possible) |

By collecting `Future` objects in a list and calling `.result()` only after all functions have been submitted, you allow the executor to run functions **concurrently** — the first call to `.result()` waits only as long as needed for that specific function, and any function that finished earlier is retrieved instantly.

This pattern — submit all work first, collect results later — is the foundation of every `executorlib` workflow.


## Executor Types

`executorlib` currently provides five executor classes. They share the same `submit()` / `map()` / `Future` API but differ in *how* they dispatch work and *where* that work runs:

| Name | Communication | Application | Internal command | Best used for |
|------|---------------|-------------|------------------|---------------|
| `SingleNodeExecutor` | Socket | Workstation / laptop | `Popen()` | Development and testing |
| `SlurmClusterExecutor` | Files | HPC login node | `sbatch` | Long-running jobs that outlive your session |
| `SlurmJobExecutor` | Socket | Inside a SLURM job | `srun` | Many short tasks within one allocation |
| `FluxClusterExecutor` | Files | HPC / Flux instance | `flux submit` | Long-running jobs; checkpointing |
| `FluxJobExecutor` | Socket | Inside a Flux job | `flux run` | High-throughput, low-latency task dispatch |

**Cluster vs. Job executors — when to use which:**

- The **Cluster executors** (`SlurmClusterExecutor`, `FluxClusterExecutor`) submit each Python function as a *new* scheduler job. The Python process that called `submit()` can be stopped while the functions are still in the queue or running; when the Python process is restarted the cached results are reloaded automatically. Choose these when your functions take minutes to hours.

- The **Job executors** (`SlurmJobExecutor`, `FluxJobExecutor`) run *inside an existing allocation* and communicate with worker processes over sockets. This avoids the scheduler overhead of submitting one job per function call. Choose these when your functions are short (seconds to minutes) and you want low latency.

- The **`SingleNodeExecutor`** mimics the `FluxJobExecutor` / `SlurmJobExecutor` behaviour but runs entirely on your local machine without any scheduler. Use it to develop and test workflows on a laptop before scaling to the cluster.

> **Recommended workflow:** Write and debug code using `SingleNodeExecutor`, then swap the > class name to `FluxJobExecutor` or `FluxClusterExecutor` when submitting to the cluster. > No other code changes are required.

In the following sections the `SingleNodeExecutor`, `FluxClusterExecutor`, and `FluxJobExecutor` are demonstrated. The SLURM equivalents (`SlurmClusterExecutor`, `SlurmJobExecutor`) behave identically and are omitted here because SLURM is not available in this demonstration environment.


## SingleNodeExecutor

The `SingleNodeExecutor` is designed for **developing and testing HPC workflows on your workstation** — no scheduler required. Internally it spawns worker processes with Python's `subprocess.Popen`, similar to `ProcessPoolExecutor`, but it also supports all of the advanced features introduced later in this notebook (caching, `resource_dict`, `block_allocation`, nested executors, etc.).

**Key principle: write once, run anywhere.**
Because every `executorlib` executor shares the same API, you can develop a workflow with `SingleNodeExecutor` on your laptop and scale it to a thousand-core HPC system by changing only the executor class name — the function definitions and the `submit()` calls remain identical.


In [4]:
from executorlib import SingleNodeExecutor

with SingleNodeExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


## FluxClusterExecutor

The `FluxClusterExecutor` submits each Python function as an **individual Flux job** using `flux submit`. It is designed to be called from the **login node** (or from inside a parent Flux instance) and uses the **filesystem** to exchange data between the main Python process and the worker processes.

**File-based communication advantages:**
- The Python session that called `submit()` can be **stopped** (e.g. your SSH session disconnects or you close the notebook) while submitted functions are still queued or running. When you restart Python the futures are automatically reloaded from the cache.
- Dependencies between functions are expressed as futures and communicated directly to the Flux scheduler, so dependent jobs are queued but only released when their inputs are ready.

**When to choose `FluxClusterExecutor`:**
Use it when individual function calls are long (minutes to hours), when you want the resilience of filesystem-based checkpointing, or when you need to manage scheduler-level dependencies between tasks.


In [5]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


In [6]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒ7NTi2F9 jovyan   executorl+ CD      1      1   1.182s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7KNKX9R jovyan   executorl+ CD      1      1   1.175s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7GFT2mM jovyan   executorl+ CD      1      1   1.211s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7D1AbyZ jovyan   executorl+ CD      1      1   1.197s jupyter-janj-lanl-executorlib-tutorial-5rv11ome


### Cache

The `FluxClusterExecutor` (and `SlurmClusterExecutor`) persist every function call and its result to an HDF5 file in a *cache directory*. This enables two important properties:

1. **Resilience:** if the Python session is interrupted the completed results are not lost.
2. **Idempotency:** if the same function is called with the same arguments a second time, the cached result is returned immediately without re-submitting to the scheduler.

The cache directory is set with the `cache_directory` parameter (default: `./executorlib_cache`). To inspect the contents of the cache programmatically, use `get_cache_data()`:


In [7]:
import pandas
from executorlib import get_cache_data

pandas.DataFrame(get_cache_data(cache_directory="executorlib_cache"))

,function,input_args,input_kwargs,output,resource_dict,runtime,queue_id,error_log_file,filename
0,<built-in function sum>,"[[1, 1]]",{},2,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001330,236290310144,None,/home/jovyan/executorlib_cache/sum5684c948e337...
1,<function calc_add at 0x7f8e77252de0>,"[2, <executorlib.task_scheduler.file.shared.Fu...",{},3,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001926,382956732416,None,/home/jovyan/executorlib_cache/calc_add0a7cba1...
2,<function calc_add at 0x7f8dc717b2e0>,"[1, 0]",{},1,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001483,380742139904,None,/home/jovyan/executorlib_cache/calc_add2c62f7d...
3,<built-in function sum>,"[[3, 3]]",{},6,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001356,240467836928,None,/home/jovyan/executorlib_cache/sumc17e02a6dc40...
4,<function calc_add at 0x7f8dc717a520>,"[3, <executorlib.task_scheduler.file.shared.Fu...",{},6,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001995,384986775552,None,/home/jovyan/executorlib_cache/calc_add08f0442...
5,<built-in function sum>,"[[4, 4]]",{},8,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001412,242497880064,None,/home/jovyan/executorlib_cache/sumf3976430477e...
6,<built-in function sum>,"[[2, 2]]",{},4,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001308,238421016576,None,/home/jovyan/executorlib_cache/sum3d89b9dcf89b...


### Submission Template

By default, `executorlib` submits jobs with a minimal resource specification. For the `FluxClusterExecutor` this is usually sufficient because Flux uses a flexible resource model. However, the `SlurmClusterExecutor` typically **requires** a custom submission template because SLURM is configured differently on every cluster (different partition names, QOS policies, memory limits, module loads, etc.).

Internally `executorlib` uses [pysqa](https://pysqa.readthedocs.io/) — the Python Simple Queuing System Adapter — to render job scripts from [Jinja2](https://jinja.palletsprojects.com/) templates. The template variables (`{{job_name}}`, `{{cores}}`, etc.) are filled in automatically from the `resource_dict` passed to `submit()`.

The template below demonstrates the pattern for Flux. For SLURM you would replace the `# flux:` comment directives with `#SBATCH` lines:


In [8]:
submission_template = """\
#!/bin/bash
# flux: --job-name={{job_name}}
# flux: --env=CORES={{cores}}
# flux: --output=time.out
# flux: --error=error.out
# flux: --nslots={{cores}}
{%- if run_time_max %}
# flux: --time-limit={{run_time_max}}s
{%- endif %}
{%- if dependency_list %}
{%- for jobid in dependency_list %}
# flux: --dependency=afterok:{{jobid}}
{%- endfor %}
{%- endif %}

{{command}}
"""

In [9]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i], resource_dict={"submission_template": submission_template}))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


When the **same function is called with the same arguments** the cached result is returned immediately — even if the `resource_dict` has changed. This means you can safely re-run a notebook cell or restart a workflow without duplicating computation. The cache key is determined by the function's source code, its name, and its serialised input arguments.

> **Important:** The cache is **not** automatically cleared between runs. Call > `shutil.rmtree(cache_directory)` (as shown in the Workflows notebook) or change the > `cache_directory` path when you want to force recomputation.


### MPI-parallel functions

`executorlib` allows you to assign computing resources to **individual Python functions** using the `resource_dict` argument to `submit()`. This means different functions in the same workflow can use different amounts of parallelism — for example, a preprocessing step might run serially while the main computation uses 16 MPI ranks.

The `resource_dict` accepts the following keys:

| Key | Type | Description |
|---|---|---|
| `"cores"` | `int` | Number of MPI tasks (CPU cores) for this function |
| `"threads_per_core"` | `int` | OpenMP threads per MPI rank |
| `"gpus_per_task"` | `int` | GPUs assigned to this function |
| `"mem_per_worker"` | `str` | Memory per worker, e.g. `"4GB"` |
| `"cwd"` | `str` | Working directory for the function's process |
| `"conda_environment_name"` | `str` | Conda environment to activate |
| `"error_log_file"` | `str` | File to capture errors (see Developer notebook) |

A full reference is available in the [executorlib troubleshooting documentation](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).

The example below submits an MPI-parallel function that uses `mpi4py` to discover how many ranks were allocated:


In [10]:
def calc_mpi(i):
    from mpi4py import MPI

    size = MPI.COMM_WORLD.Get_size()
    rank = MPI.COMM_WORLD.Get_rank()
    return i, size, rank

In [11]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor(pmi_mode="pmix") as exe:
    fs = exe.submit(calc_mpi, 3, resource_dict={"cores": 2})
    print(fs.result())

[(3, 2, 0), (3, 2, 1)]


The key advantage of per-function resource assignment is that **parallelism is introduced exactly where it is needed, without restructuring the rest of the workflow**. A typical scientific workflow has a mixture of:
- Serial pre/post-processing steps (data loading, plotting, analysis)
- Parallel compute-intensive steps (DFT calculations, Monte Carlo sampling, molecular dynamics)

With `executorlib` you annotate only the parallel functions with a `resource_dict`. The serial parts of the code are untouched. This is much simpler than the alternative — writing the entire workflow in MPI or splitting it across multiple SLURM scripts.


## FluxJobExecutor

The `FluxJobExecutor` (and its SLURM equivalent `SlurmJobExecutor`) runs **inside an existing Flux allocation** and communicates with worker processes over **network sockets** rather than through files. This has two consequences:

1. **Lower latency:** socket messages are delivered in microseconds, compared to milliseconds for filesystem reads/writes. This matters when submitting thousands of short tasks.
2. **No persistence:** if the Python session is interrupted, in-flight results are lost (unlike the `FluxClusterExecutor`). The session must remain alive for the duration of the workflow.

**When to choose `FluxJobExecutor`:**
Use it for high-throughput workflows with many short tasks (seconds to minutes each) where low overhead is important and you can guarantee the Python session will not be interrupted.


In [12]:
from executorlib import FluxJobExecutor

with FluxJobExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


In [13]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒGgpiFeo jovyan   python     CD      1      1   0.164s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGgoEGNT jovyan   python     CD      1      1   0.157s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGeDVWud jovyan   python     CD      1      1   0.164s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGeDVWue jovyan   python     CD      1      1   0.158s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒEZK45xb jovyan   executorl+ CD      1      1   2.620s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7NTi2F9 jovyan   executorl+ CD      1      1   1.182s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7KNKX9R jovyan   executorl+ CD      1      1   1.175s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7GFT2mM jovyan   executorl+ CD      1      1   1.211s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7D1AbyZ jovyan   executorl+ CD      1      1   1.197s jupyter-janj-lanl-exe

### Block allocation

By default, each function submitted to the `FluxJobExecutor` is dispatched to a *fresh* worker process that is started, runs the function, and then exits. This one-process-per-function model is simple but has overhead: Python imports, initialisation code, and any large data structures must be loaded anew for every call.

**Block allocation** (`block_allocation=True`) changes this: a pool of persistent worker processes is created at startup and each worker handles **multiple function calls in sequence**. Think of it as the difference between hiring a new contractor for every task (default) vs. hiring a full-time employee who handles tasks one after another (block allocation).

Benefits:
- Eliminates per-function Python startup overhead
- Allows pre-loading large datasets once and reusing them across many function calls (see the *Initialization function* section below)

Constraints:
- All functions submitted in block-allocation mode share the same resource allocation (set via `max_workers`)
- The functions must be safe to run sequentially in the same process (no hidden global state mutations between calls)


In [14]:
from executorlib import FluxJobExecutor

with FluxJobExecutor(block_allocation=True, max_workers=1) as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


In [15]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒJfpgYWB jovyan   python     CD      1      1   0.169s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGgpiFeo jovyan   python     CD      1      1   0.164s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGgoEGNT jovyan   python     CD      1      1   0.157s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGeDVWud jovyan   python     CD      1      1   0.164s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒGeDVWue jovyan   python     CD      1      1   0.158s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒEZK45xb jovyan   executorl+ CD      1      1   2.620s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7NTi2F9 jovyan   executorl+ CD      1      1   1.182s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7KNKX9R jovyan   executorl+ CD      1      1   1.175s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ7GFT2mM jovyan   executorl+ CD      1      1   1.211s jupyter-janj-lanl-exe

Block allocation is especially valuable when sampling a large parameter space with many short, similar function calls — for example, evaluating an energy function at thousands of atomic configurations, running a parameter sweep, or performing cross-validation in machine learning. The overhead of process startup and teardown is paid once per worker rather than once per function call.


In [16]:
import pandas
from executorlib import FluxJobExecutor

df = pandas.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

def sum_df(a, b):
    return a + b

with FluxJobExecutor(block_allocation=True, max_workers=1) as exe:
    df["c"] = list(exe.map(sum_df, df["a"], df["b"]))
    print(df)

   a  b  c
0  1  4  5
1  2  5  7
2  3  6  9


### Initialization function

Block allocation reuses worker processes across multiple function calls. The `init_function` parameter takes advantage of this by running **setup code once when the worker starts** and making the result available to all subsequent function calls via keyword arguments.

Typical uses:
- **Pre-loading large datasets:** read a multi-gigabyte file once and pass its handle to every function call, avoiding repeated I/O.
- **Establishing connections:** open a database connection, simulation handle, or external process once and reuse it.
- **Pre-computing expensive setup:** initialise an expensive model or data structure once.

The `init_function` must return a **dictionary**. Each key in that dictionary becomes an additional keyword argument in every function submitted to the executor. In the example below, `init_file_handle()` opens a file and returns `{"file_handle": <iterator>}`, which means every submitted function automatically receives a `file_handle` keyword argument.


In [17]:
with open("large.txt", "w") as f:
    for i in range(100):
        f.writelines(str(i) + "\n")

In [18]:
def init_file_handle():
    def open_large_file():
        with open("large.txt", "r") as f:
            for line in f:
                yield line.rstrip()
    return {"file_handle": open_large_file()}

In [19]:
def read_next_n_lines(file_handle, n=5):
    return [next(file_handle) for i in range(n)]  

In [20]:
with FluxJobExecutor(block_allocation=True, max_workers=1, init_function=init_file_handle) as exe:
    f1 = exe.submit(read_next_n_lines)
    f2 = exe.submit(read_next_n_lines, n=10)
    print(f1.result(), f2.result())

['0', '1', '2', '3', '4'] ['5', '6', '7', '8', '9', '10', '11', '12', '13', '14']


### Nested Executors

`executorlib` supports creating additional executor instances **inside the functions that are already submitted to an executor** — a pattern called *nested executors*. This allows you to express hierarchical parallelism: the outer executor distributes coarse-grained tasks across nodes, and each task internally uses another executor to distribute fine-grained sub-tasks.


The hierarchical nature of the Flux scheduler makes nested executors possible: a function running inside a Flux job can itself spawn further Flux sub-jobs using the same `flux run` / `flux submit` mechanism, as long as Flux nesting is enabled with `flux_executor_nesting=True`.

**Why use nested executors?**

Consider a workflow where you have 10 independent experiments, and each experiment internally performs 4 parallel simulation steps. You could flatten this into 40 individual submissions, but that mixes the experiment-level and step-level logic. With nested executors you express it naturally:

```
outer FluxJobExecutor
  └─ function: run_experiment(i)
       └─ inner FluxJobExecutor
            ├─ submit: step_a()
            ├─ submit: step_b()
            ├─ submit: step_c()
            └─ submit: step_d()
```

**Requirements:**
- Nesting must be explicitly enabled: `FluxJobExecutor(flux_executor_nesting=True)`
- The inner executor must also be a `FluxJobExecutor` and must request cores via `resource_dict={"cores": N}`
- This feature is only available with Flux (not SLURM) because SLURM does not support sub-job hierarchies within an allocation

The example below demonstrates this: `calc_nested` creates an inner `FluxJobExecutor` inside a function that is itself submitted to an outer executor.


In [21]:
def calc_nested():
    from executorlib import FluxJobExecutor

    with FluxJobExecutor(pmi_mode="pmix") as exe:
        fs = exe.submit(sum, [1, 1])
        return fs.result()

In [22]:
with FluxJobExecutor(pmi_mode="pmix", flux_executor_nesting=True) as exe:
    fs = exe.submit(calc_nested)
    print(fs.result())

2
